# 02 — Clean and Construct Variables**Goals**1. Recode NLS missing codes (-1, -2, -3, -7) to `NaN`2. Build a binary SNAP/food-stamp exposure indicator: did the respondent ever receive food stamps 1994–2006?3. Build a binary TANF/AFDC exposure indicator: did the respondent ever receive AFDC 1994–1998?4. Combine into a 4-level program group: `Neither / SNAP only / TANF only / Both`5. Build a 3-level education outcome variable: `No HS / HS or GED / Has professional degree`6. Save a cleaned analysis dataset for notebook 03

In [ ]:
import pandas as pdimport numpy as npfrom pathlib import PathPROJECT = Path('..').resolve()DATA    = PROJECT / 'data' / 'nlscya_subset.csv'OUT_DIR = PROJECT / 'data'df = pd.read_csv(DATA)print('Loaded:', df.shape)

## Step 1. Recode NLS missing codes to NaNNLS uses:- `-1` = Refused- `-2` = Don't Know- `-3` = Invalid skip- `-4` = Valid skip- `-5` = Non-interview- `-7` = Missing / not asked in this roundAll of these mean "no usable value" → recode to `NaN`.

In [ ]:
missing_codes = [-1, -2, -3, -4, -5, -7]for col in df.columns:    if col in ('CPUBID', 'MPUBID'):        continue    df[col] = df[col].replace(missing_codes, np.nan)# Verify: no more negativesprint('Min values across columns (post-recode, ignoring NaN):')print(df.drop(columns=['CPUBID', 'MPUBID']).min())

## Step 2. SNAP/food-stamp exposure (1994–2006)Following the Milestone 1 framing, a respondent is "SNAP-exposed" if they reported receiving food stamps in **any** wave from 1994–2006. NLS food-stamp items are yes (1) / no (0).A child counts as SNAP-exposed if at least one wave answered "Yes" AND no waves are uniformly missing.

In [ ]:
snap_cols = ['FOODSTAMPS_1994', 'FOODSTAMPS_1996', 'FOODSTAMPS_1998',             'FOODSTAMPS_2000', 'FOODSTAMPS_2002', 'FOODSTAMPS_2004',             'FOODSTAMPS_2006']# snap_ever = 1 if any 1, 0 if all 0, NaN if all missingdf['snap_ever'] = df[snap_cols].max(axis=1)print('snap_ever distribution:')print(df['snap_ever'].value_counts(dropna=False))

## Step 3. TANF/AFDC exposure (1994–1998)AFDC was the welfare program that TANF replaced in 1997. The CYA dataset asks about AFDC receipt in 1994, 1996, and 1998 — so we measure AFDC exposure across those three waves.This is a narrower window than SNAP exposure, but it is the cleanest welfare-cash-assistance signal available in this public-use file.

In [ ]:
afdc_cols = ['AFDC_1994', 'AFDC_1996', 'AFDC_1998']df['tanf_ever'] = df[afdc_cols].max(axis=1)print('tanf_ever (AFDC) distribution:')print(df['tanf_ever'].value_counts(dropna=False))

## Step 4. Four-group exposure variable (Unit 4: user-defined function)

In [ ]:
def assign_group(row):    """Assign each respondent to a SNAP/TANF exposure group.    Returns NaN if either exposure indicator is missing."""    s = row['snap_ever']    t = row['tanf_ever']    if pd.isna(s) or pd.isna(t):        return np.nan    if s == 1 and t == 1: return 'Both SNAP and TANF'    if s == 1 and t == 0: return 'SNAP only'    if s == 0 and t == 1: return 'TANF only'    return 'Neither'df['program_group'] = df.apply(assign_group, axis=1)print('program_group distribution:')print(df['program_group'].value_counts(dropna=False))

## Step 5. Education outcome (3-level ordinal)NLS-CYA gives us cross-round (XRND) summary indicators for each respondent:- `HSDIPLOMA_EVER` — ever received HS diploma (1/0)- `GED_EVER` — ever received GED (1/0)- `PROFDEGREE_EVER` — ever received a professional/college degree (1/0)We build an ordinal:- 0 = No HS diploma or GED- 1 = HS diploma or GED (but no professional degree)- 2 = Has professional / college degree

In [ ]:
def edu_level(row):    h, g, p = row['HSDIPLOMA_EVER'], row['GED_EVER'], row['PROFDEGREE_EVER']    if pd.isna(h) and pd.isna(g):        return np.nan    has_hs   = (h == 1) or (g == 1)    has_prof = (p == 1)    if has_prof: return 2    if has_hs:   return 1    return 0df['edu_level'] = df.apply(edu_level, axis=1)df['edu_label'] = df['edu_level'].map({0: 'No HS / GED',                                       1: 'HS or GED',                                       2: 'Professional degree'})print('edu_level distribution:')print(df['edu_label'].value_counts(dropna=False))

## Step 6. Save the cleaned analysis dataset

In [ ]:
out = OUT_DIR / 'nlscya_cleaned.csv'df.to_csv(out, index=False)print(f'Saved: {out.name}  ({out.stat().st_size/1e6:.2f} MB)')print(f'Columns: {list(df.columns)}')

Move on to `03_analyze_visualize.ipynb`.